In [3]:
!pip install -q datasets pandas Sastrawi

In [4]:
from datasets import load_dataset
import pandas as pd

# Mengambil dataset dari Hugging Face
dataset = load_dataset(
    "dipawidia/ecommerce-product-reviews-sentiment",
    split="train"
)

# Mengubah dataset menjadi tabel Pandas
df = dataset.to_pandas()

# Menampilkan 5 baris pertama
display(df.head())

README.md:   0%|          | 0.00/1.51k [00:00<?, ?B/s]

labeledReview.datasetFix.json:   0%|          | 0.00/3.15M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/11606 [00:00<?, ? examples/s]

,review,translate,sentimen
0,Suara dan mic puas. Hanya saya pesan yg nyala ...,sound and mic satisfied. I only ordered the on...,0
1,Good. Mudah-mudahan awet. Makasih,good. hopefully durable. Thanks,1
2,"kondisi dus robek2, tolong di perhatikan lagi","the condition of the box is torn, please pay a...",0
3,Kualitas produk baik. Harga terjangkau dan pen...,product quality is good. affordable prices and...,0
4,"Overall bagus, tombol berfungsi semuanya, tapi...","overall is good, the buttons all work, but the...",0


In [5]:
# 1. Menampilkan jumlah baris dan kolom
print("Jumlah baris dan kolom:", df.shape)
print("Jumlah data:", df.shape[0])
print("Jumlah kolom:", df.shape[1])

# 2. Menampilkan nama kolom
print("\nNama kolom:")
print(df.columns.tolist())

# 3. Menampilkan tipe data dan informasi setiap kolom
print("\nInformasi dataset:")
df.info()

# 4. Memeriksa nilai kosong
print("\nJumlah nilai kosong:")
print(df.isnull().sum())

# 5. Menampilkan jumlah label sentimen
print("\nDistribusi label sentimen:")
print(df["sentimen"].value_counts().sort_index())

Jumlah baris dan kolom: (11606, 3)
Jumlah data: 11606
Jumlah kolom: 3

Nama kolom:
['review', 'translate', 'sentimen']

Informasi dataset:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 11606 entries, 0 to 11605
Data columns (total 3 columns):
 #   Column     Non-Null Count  Dtype 
---  ------     --------------  ----- 
 0   review     11606 non-null  object
 1   translate  11606 non-null  object
 2   sentimen   11606 non-null  int64 
dtypes: int64(1), object(2)
memory usage: 272.1+ KB

Jumlah nilai kosong:
review       0
translate    0
sentimen     0
dtype: int64

Distribusi label sentimen:
sentimen
0    5803
1    5803
Name: count, dtype: int64


In [7]:
def preprocessing(teks):
    # Tahap 1: Mengubah huruf menjadi kecil
    teks = case_folding(teks)

    # Tahap 2: Membersihkan teks
    teks = cleaning(teks)

    # Tahap 3: Memisahkan teks menjadi kata
    tokens = tokenisasi(teks)

    # Tahap 4: Menghapus stopword
    tokens = stopword_removal(tokens)

    # Tahap 5: Mengubah kata menjadi bentuk dasar
    tokens = stemming(tokens)

    # Menggabungkan kata menjadi kalimat
    return " ".join(tokens)

print("Fungsi preprocessing berhasil dibuat.")


Fungsi preprocessing berhasil dibuat.


In [9]:
import re
from Sastrawi.Stemmer.StemmerFactory import StemmerFactory
from Sastrawi.StopWordRemover.StopWordRemoverFactory import (
    StopWordRemoverFactory
)

# Membuat stemmer
stemmer = StemmerFactory().create_stemmer()

# Membuat daftar stopword bahasa Indonesia
stopword_factory = StopWordRemoverFactory()
stopwords = set(stopword_factory.get_stop_words())

# Mempertahankan kata negasi
kata_negasi = {"tidak", "bukan", "belum", "jangan"}
stopwords = stopwords - kata_negasi


# 1. Case folding
def case_folding(teks):
    return str(teks).lower()


# 2. Cleaning
def cleaning(teks):
    teks = re.sub(r"https?://\S+|www\.\S+", " ", teks)
    teks = re.sub(r"\d+", " ", teks)
    teks = re.sub(r"[^a-zA-Z\s]", " ", teks)
    teks = re.sub(r"\s+", " ", teks).strip()
    return teks


# 3. Tokenisasi
def tokenisasi(teks):
    return teks.split()


# 4. Stopword removal
def stopword_removal(tokens):
    return [kata for kata in tokens if kata not in stopwords]


# 5. Stemming
def stemming(tokens):
    return [stemmer.stem(kata) for kata in tokens]


# Menggabungkan seluruh proses
def preprocessing(teks):
    teks = case_folding(teks)
    teks = cleaning(teks)
    tokens = tokenisasi(teks)
    tokens = stopword_removal(tokens)
    tokens = stemming(tokens)
    return " ".join(tokens)


print("Semua fungsi preprocessing berhasil dibuat.")

Semua fungsi preprocessing berhasil dibuat.


In [10]:
# Menghapus baris yang tidak memiliki ulasan
df = df.dropna(subset=["review"]).copy()

# Memastikan semua ulasan berbentuk teks
df["review"] = df["review"].astype(str)

# Menjalankan preprocessing pada setiap ulasan
df["review_preprocessing"] = df["review"].apply(preprocessing)

# Menampilkan hasil preprocessing
display(
    df[["review", "review_preprocessing", "sentimen"]].head()
)

,review,review_preprocessing,sentimen
0,Suara dan mic puas. Hanya saya pesan yg nyala ...,suara mic puas pesan yg nyala sy pasti lg klik...,0
1,Good. Mudah-mudahan awet. Makasih,good mudah mudah awet makasih,1
2,"kondisi dus robek2, tolong di perhatikan lagi",kondisi dus robek perhati,0
3,Kualitas produk baik. Harga terjangkau dan pen...,kualitas produk baik harga jangkau kirim cepet...,0
4,"Overall bagus, tombol berfungsi semuanya, tapi...",overall bagus tombol fungsi semua packaging ku...,0


In [11]:
# Mengambil 10 ulasan pertama
contoh = df[
    ["review", "review_preprocessing", "sentimen"]
].head(10).copy()

# Mengubah angka label menjadi nama sentimen
contoh["sentimen"] = contoh["sentimen"].map({
    0: "Negatif",
    1: "Positif"
})

# Mengganti nama kolom agar lebih mudah dipahami
contoh = contoh.rename(columns={
    "review": "Teks Sebelum Preprocessing",
    "review_preprocessing": "Teks Sesudah Preprocessing",
    "sentimen": "Label Sentimen"
})

# Menampilkan tabel perbandingan
display(contoh)

,Teks Sebelum Preprocessing,Teks Sesudah Preprocessing,Label Sentimen
0,Suara dan mic puas. Hanya saya pesan yg nyala ...,suara mic puas pesan yg nyala sy pasti lg klik...,Negatif
1,Good. Mudah-mudahan awet. Makasih,good mudah mudah awet makasih,Positif
2,"kondisi dus robek2, tolong di perhatikan lagi",kondisi dus robek perhati,Negatif
3,Kualitas produk baik. Harga terjangkau dan pen...,kualitas produk baik harga jangkau kirim cepet...,Negatif
4,"Overall bagus, tombol berfungsi semuanya, tapi...",overall bagus tombol fungsi semua packaging ku...,Negatif
5,"Produk ini original, Dan kualitasnya sangat ba...",produk original kualitas sangat baik cepat kir...,Positif
6,kualitas sangat jelek. bocor,kualitas sangat jelek bocor,Negatif
7,"terimakasih banyak ka ya, barang sudah diterim...",terimakasih banyak ka barang terima baik sukse...,Positif
8,"Body nya lebih berat dari cm 1579, spiral nya ...",body nya lebih berat cm spiral nya lebih tipis...,Negatif
9,pesanan di terima tanggal 27 January 2021 teta...,pesan terima tanggal january baru buka pagi ta...,Negatif


In [12]:
# Menyimpan dataset hasil preprocessing
nama_file = "hasil_preprocessing_sentimen.csv"

df.to_csv(
    nama_file,
    index=False,
    encoding="utf-8-sig"
)

print("Dataset berhasil disimpan:", nama_file)

# Mengunduh file dari Google Colab
from google.colab import files
files.download(nama_file)

Dataset berhasil disimpan: hasil_preprocessing_sentimen.csv


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>